In [7]:
import torch
from torch import nn
import math
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import random
import time
import mesa
import random
random.seed(0)
import warnings
warnings.filterwarnings("ignore")
import sys
import numpy as np

sys.path.append("C:/Users/met48/Desktop/TS-Clustering/mesa-examples-main/mesa-examples-main/examples/forest_fire/") 
from forest_fire import __init__ as forestFireInit
from forest_fire import agent as forestFireAgent
from forest_fire import model as forestFireModel

sys.path.append("C:/Users/met48/Desktop/TS-Clustering/mesa-examples-main/mesa-examples-main/examples/bank_reserves/") 
from bank_reserves import random_walk as bankReservesRandomWalk
from bank_reserves import agents as bankReservesAgent
from bank_reserves import model as bankReservesModel

sys.path.append("C:/Users/met48/Desktop/TS-Clustering/mesa-examples-main/mesa-examples-main/examples/epstein_civil_violence/") 
from epstein_civil_violence import __init__ as epsteinInit
from epstein_civil_violence import agent as epsteinCVAgent
from epstein_civil_violence import model as epsteinCVModel

In [8]:
n_samples = 10

# Bank Reserves

In [29]:
inputs = list()
for i in np.arange(n_samples):
    reserve_perc = random.uniform(0,100)
    inputs.append(reserve_perc)

## Actual ABM

In [30]:
br_actual_times = []
for i in np.arange(n_samples):
    start = time.perf_counter()
    bankRes = bankReservesModel.BankReserves(init_people=500, rich_threshold=10, reserve_percent=inputs[i])
    bankRes.run_model()
    results = bankRes.datacollector.get_model_vars_dataframe()
    end = time.perf_counter()
    br_actual_times.append(end-start)

In [31]:
np.mean(br_actual_times)

0.4220971799999461

## Surrogate ABM

In [32]:
class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=100, condition_dim=1, output_dim=101):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
        self.temporal_smoother = nn.Conv1d(1, 1, kernel_size=9, padding=4)  # keeps shape

    def forward(self, z, conditions):
        x = torch.cat([z, conditions], dim=1)
        out = self.fc(x)
        out = out.unsqueeze(1)  # (B, 1, 252)
        out = self.temporal_smoother(out)
        return out.squeeze(1)

generator = ConditionalGenerator()

In [33]:
br_surrogate_times = []
generator = ConditionalGenerator()
generator.load_state_dict(torch.load("C:/Users/met48/Desktop/ABM-Surrogate/generator_gan_br_ts_128_dupe.pth"))
generator.eval()
with torch.no_grad():
    for i in np.arange(n_samples):
        latent_dim = 100  # or whatever your latent size is
        z = torch.randn(1, latent_dim)
        cond = torch.tensor([[inputs[i]]], dtype=torch.float32)
        start = time.perf_counter()
        generated_samples = generator(z, cond)  # (n_samples, 252)
        end = time.perf_counter()
        br_surrogate_times.append(end-start)

In [34]:
np.mean(br_surrogate_times)

0.0012392299997372902

# Epstein

In [ ]:
inputs = []
for i in np.arange(n_samples):
    density = random.uniform(0, 1)
    inputs.append[density]

## Actual ABM

In [ ]:
ecv_actual_times = []
for i in np.arange(n_samples):
    density = random.uniform(0, 1)
    start = time.perf_counter()
    cit_dens = random.uniform(0.5, 1)
    cop_dens = random.uniform(0, 1-cit_dens)
    leg = random.uniform(0, 1)
    bankRes = epsteinCVModel.EpsteinCivilViolence(citizen_density=cit_dens, cop_density=cop_dens, citizen_vision=5, cop_vision=5, legitimacy=leg, max_jail_term=30, active_threshold = 0.1, arrest_prob_constant = 2.3, movement=True, max_iters=250)
    bankRes.run_model()
    results = bankRes.datacollector.get_model_vars_dataframe()
    end = time.perf_counter()
    ecv_actual_times.append(end-start)

## Surrogate ABM

In [ ]:
class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=100, condition_dim=3, output_dim=252):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
        self.temporal_smoother = nn.Conv1d(1, 1, kernel_size=5, padding=2)  # keeps shape

    def forward(self, z, conditions):
        x = torch.cat([z, conditions], dim=1)
        out = self.fc(x)
        out = out.unsqueeze(1)  # (B, 1, 252)
        out = self.temporal_smoother(out)
        return out.squeeze(1)

In [ ]:
ecv_surrogate_times = []
generator = ConditionalGenerator()
generator.load_state_dict(torch.load("C:/Users/met48/Desktop/ABM-Surrogate/generator_gan_ecv_ts_128_5.pth"))
generator.eval()
with torch.no_grad():
    for i in np.arange(n_samples):
        latent_dim = 100  # or whatever your latent size is
        z = torch.randn(1, latent_dim)
        cond = torch.tensor([[inputs[i]]], dtype=torch.float32)
        start = time.perf_counter()
        generated_samples = generator(z, cond)  # (n_samples, 252)
        end = time.perf_counter()
        ecv_surrogate_times.append(end-start)

# Forest Fire

In [37]:
inputs = []
for i in np.arange(n_samples):
    density = random.uniform(0, 1)
    inputs.append(density)

## Actual ABM

In [38]:
ff_actual_times = []
for i in np.arange(n_samples):
    start = time.perf_counter()
    fire = forestFireModel.ForestFire(100, 100, inputs[i])
    fire.run_model()
    results = fire.datacollector.get_model_vars_dataframe()
    end = time.perf_counter()
    ff_actual_times.append(end-start)

AttributeError: module 'mesa' has no attribute 'time'

## Surrogate ABM

In [ ]:
class ConditionalGenerator(nn.Module):
    def __init__(self, latent_dim=100, condition_dim=1, output_dim=155):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + condition_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        )
        self.temporal_smoother = nn.Conv1d(1, 1, kernel_size=9, padding=4)  # keeps shape

    def forward(self, z, conditions):
        x = torch.cat([z, conditions], dim=1)
        out = self.fc(x)
        out = out.unsqueeze(1)  # (B, 1, 252)
        out = self.temporal_smoother(out)
        return out.squeeze(1)